# osu! Chat Bot RAG in Google Colab

This notebook is a guided version of the command-line workflow for the osu! chat bot prototype. It is designed for Google Colab, but most cells also work in a local Jupyter notebook.

The goal is to make the pipeline easy to inspect and rerun without needing your own laptop to do the heavy work.

The project currently behaves like a RAG system rather than a model fine-tuning project. The important stages are:

1. Parse the osu! wiki/news corpus into structured documents.
2. Split those documents into retrieval chunks.
3. Build a deterministic terminology dictionary.
4. Optionally extract extra entity candidates with GLiNER.
5. Evaluate retrieval quality on seed questions.
6. Optionally build a dense Qdrant index with sentence-transformer embeddings.

The notebook defaults to conservative, low-cost settings. Dense indexing and GLiNER entity extraction are optional because they can take longer and may benefit from a GPU runtime.

## 0. Colab Runtime Recommendation

For the first run, use a standard CPU runtime. That is enough for ingesting, validating, keyword retrieval, and the seed evaluation.

Use a GPU runtime only when you want to try one of these optional stages:

- `osu-bot index` with `device = "cuda"` for faster embeddings.
- `osu-bot entities` with GLiNER over many chunks.

In Colab, change runtime hardware from `Runtime -> Change runtime type -> Hardware accelerator`.

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Running in Colab: {IN_COLAB}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except Exception as exc:
    print(f"PyTorch check skipped: {exc}")

## 1. Get the Project Code

Colab opens a notebook by itself; it does not automatically clone the whole repository that contains the notebook. Set `REPO_URL` to your GitHub repository URL before running this cell.

Example:

```python
REPO_URL = "https://github.com/your-name/osu-chat-bot.git"
```

If you are running this notebook locally from the project root, leave `REPO_URL` empty.

In [ ]:
# TODO: In Colab, set this to your repository URL.
REPO_URL = ""

PROJECT_DIR = Path("/content/osu-chat-bot") if IN_COLAB else Path.cwd()

if IN_COLAB and not (PROJECT_DIR / "pyproject.toml").exists():
    if not REPO_URL:
        raise ValueError(
            "Set REPO_URL to your GitHub repo URL, then rerun this cell. "
            "Example: REPO_URL = 'https://github.com/your-name/osu-chat-bot.git'"
        )
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")
print(f"pyproject.toml exists: {(Path.cwd() / 'pyproject.toml').exists()}")

## 2. Install Python Dependencies

The normal project dependencies are enough for the baseline RAG pipeline:

- `sentence-transformers` for dense embeddings.
- `qdrant-client` for the local vector store.
- `pytest` from the `dev` extra, useful if you want to run tests.

The GLiNER entity extraction dependency is intentionally not installed here. It is installed later only if you choose to run that optional section.

In [ ]:
%pip install -q -e ".[dev]"

## 3. Download the osu! Wiki Corpus

The chatbot reads from a local checkout of `ppy/osu-wiki`. The default config expects it at `database/osu-wiki`.

The clone is shallow to keep Colab setup quick. If you need full git history later, remove `--depth 1`.

In [ ]:
OSU_WIKI_DIR = Path("database/osu-wiki")

if not OSU_WIKI_DIR.exists():
    OSU_WIKI_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ppy/osu-wiki.git", str(OSU_WIKI_DIR)],
        check=True,
    )
else:
    print(f"Corpus already exists: {OSU_WIKI_DIR}")

print(f"Corpus path: {OSU_WIKI_DIR.resolve()}")

## 4. Write a Colab-Friendly Config

This config keeps artifacts inside the project folder so they are easy to zip and download later.

The embedding device is selected automatically:

- `cuda` when Colab has a GPU.
- `cpu` otherwise.

The local Qdrant URL stores vectors under `artifacts/rag/qdrant`. This is simple and portable for notebook experiments.

In [ ]:
try:
    import torch
    EMBEDDING_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    EMBEDDING_DEVICE = "cpu"

HF_CACHE_DIR = "/content/hf-cache" if IN_COLAB else "artifacts/hf-cache"

CONFIG_PATH = Path("config.colab.toml")
CONFIG_PATH.write_text(
    f"""
[corpus]
osu_wiki_path = "database/osu-wiki"
language = "en"
include_news = true

[artifacts]
path = "artifacts/rag"

[embedding]
model = "sentence-transformers/all-MiniLM-L6-v2"
cache_folder = "{HF_CACHE_DIR}"
device = "{EMBEDDING_DEVICE}"
local_files_only = false

[qdrant]
url = "file://artifacts/rag/qdrant"
collection = "osu_wiki_en"
vector_size = 384

[retrieval]
dense_top_k = 24
final_top_k = 6

[ollama]
url = "http://127.0.0.1:11434"
model = "mistral"
temperature = 0.1
""".strip()
    + "\n",
    encoding="utf-8",
)

print(CONFIG_PATH.read_text())

## 5. Helper Functions for Running `osu-bot`

The project exposes a CLI named `osu-bot`. In notebooks, it is useful to wrap commands in a small Python helper so every command uses the same config file.

Example command from a terminal:

```bash
osu-bot --config config.colab.toml stats
```

Equivalent notebook call:

```python
run_osu_bot("stats")
```

In [ ]:
import json
import shlex

def run_osu_bot(*args, check=True):
    cmd = ["osu-bot", "--config", str(CONFIG_PATH), *map(str, args)]
    print("$ " + " ".join(shlex.quote(part) for part in cmd))
    return subprocess.run(cmd, check=check)

def show_json(path, max_chars=4000):
    path = Path(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    text = json.dumps(data, indent=2, ensure_ascii=False)
    print(text[:max_chars])
    if len(text) > max_chars:
        print(f"\n... truncated after {max_chars} characters ...")
    return data

ARTIFACT_DIR = Path("artifacts/rag")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 6. Build the Baseline Artifacts

These commands are the core non-GPU workflow.

### `ingest`

Parses English wiki pages and news posts into structured documents and retrieval chunks.

Outputs include:

- `documents_structured.jsonl`
- `chunks_hierarchical.jsonl`
- `ingest_report.json`

### `terms`

Builds a deterministic osu!-specific terminology dictionary from titles, headings, paths, tags, and link text.

### `links`

Builds reviewable hyperlink alias artifacts. This helps inspect which link texts are useful aliases and which are noisy.

These stages are safe to run on a CPU Colab runtime.

## Background Jobs for Long Runs

Colab notebooks can become uncomfortable when a long command is tied directly to a cell. For heavier tasks, run the CLI as a background process and write all output to a log file.

Use this for dense indexing, dense evaluation, and GLiNER entity extraction. The notebook stays responsive, and you can inspect logs in small chunks.

The helper writes:

- `artifacts/rag/jobs/<job>.pid`
- `artifacts/rag/jobs/<job>.log`
- `artifacts/rag/jobs/<job>.cmd.txt`

In [ ]:
import subprocess
import shlex
from pathlib import Path

JOB_DIR = ARTIFACT_DIR / "jobs"
JOB_DIR.mkdir(parents=True, exist_ok=True)

def start_osu_bot_job(job_name, *args, pre_command=None):
    job_name = str(job_name)
    pid_path = JOB_DIR / f"{job_name}.pid"
    log_path = JOB_DIR / f"{job_name}.log"
    cmd_path = JOB_DIR / f"{job_name}.cmd.txt"

    if pid_path.exists():
        old_pid = pid_path.read_text(encoding="utf-8").strip()
        if old_pid:
            check = subprocess.run(["bash", "-lc", f"ps -p {shlex.quote(old_pid)} >/dev/null"], text=True)
            if check.returncode == 0:
                raise RuntimeError(f"Job {job_name!r} is already running with PID {old_pid}.")

    cmd = ["osu-bot", "--config", str(CONFIG_PATH), *map(str, args)]
    shell_cmd = " ".join(shlex.quote(part) for part in cmd)
    if pre_command:
        shell_cmd = f"{pre_command} && {shell_cmd}"

    cmd_path.write_text(shell_cmd + "\n", encoding="utf-8")

    launcher = (
        f"nohup bash -lc {shlex.quote(shell_cmd)} "
        f"> {shlex.quote(str(log_path))} 2>&1 & "
        f"echo $! > {shlex.quote(str(pid_path))}"
    )
    subprocess.run(["bash", "-lc", launcher], check=True)

    pid = pid_path.read_text(encoding="utf-8").strip()
    print(f"Started {job_name}: PID {pid}")
    print(f"Command: {cmd_path}")
    print(f"Log: {log_path}")
    return {"job": job_name, "pid": pid, "log": log_path, "command": cmd_path}

def job_status(job_name):
    pid_path = JOB_DIR / f"{job_name}.pid"
    if not pid_path.exists():
        print(f"No PID file for {job_name!r}.")
        return None
    pid = pid_path.read_text(encoding="utf-8").strip()
    subprocess.run(["bash", "-lc", f"ps -p {shlex.quote(pid)} -o pid,etime,cmd"])
    return pid

def show_job_log(job_name, chars=4000):
    log_path = JOB_DIR / f"{job_name}.log"
    if not log_path.exists():
        print(f"No log file for {job_name!r}: {log_path}")
        return ""
    text = log_path.read_text(encoding="utf-8", errors="replace")
    tail = text[-chars:]
    print(tail)
    return tail

def stop_job(job_name):
    pid_path = JOB_DIR / f"{job_name}.pid"
    if not pid_path.exists():
        print(f"No PID file for {job_name!r}.")
        return
    pid = pid_path.read_text(encoding="utf-8").strip()
    subprocess.run(["bash", "-lc", f"kill {shlex.quote(pid)}"])
    print(f"Stop requested for {job_name}: PID {pid}")

In [ ]:
run_osu_bot("ingest")
run_osu_bot("terms")
run_osu_bot("links")

## 7. Inspect Statistics and Validation

Run these before indexing or evaluating. They give you a quick health check.

- `stats` summarizes document, chunk, structure, and term counts.
- `validate` reports possible artifact problems.

Validation severities mean:

- `error`: fix before trusting downstream results.
- `warning`: inspect because retrieval quality may be affected.
- `info`: useful corpus notes, often acceptable.

In [ ]:
run_osu_bot("stats")

# validate returns exit code 1 if it finds errors. We keep the notebook running so you can inspect the report.
validation = run_osu_bot("validate", check=False)
print(f"validate exit code: {validation.returncode}")

In [ ]:
stats_report = show_json(ARTIFACT_DIR / "stats_report.json", max_chars=2500)
validation_report = show_json(ARTIFACT_DIR / "validation_report.json", max_chars=2500)

## 8. Try Keyword-Only Retrieval

`inspect --keyword-only` does not require Qdrant or embeddings. It uses the parsed chunks and terms, so it is the fastest way to check whether the pipeline is basically working.

Try changing the questions below. Good test questions usually mention a real osu! concept, a symptom, or a mapping/ranking term.

In [ ]:
example_questions = [
    "What is a beatmap?",
    "What does AR change in osu?",
    "How does a beatmap become ranked?",
    "osu feels choppy and my frames are awful, what should I check?",
]

for question in example_questions:
    print("\n" + "=" * 100)
    print(question)
    print("=" * 100)
    run_osu_bot("inspect", question, "--keyword-only")

## 9. Run the Seed Retrieval Evaluation

The seed dataset lives at `eval/osu_seed.jsonl`. Each row has:

- `category`: a rough question group.
- `question`: the user-style prompt.
- `expected_document_ids`: one or more wiki pages that should appear in retrieval.

Example row:

```json
{"category": "definition", "question": "wait, what even is a beatmap in osu?", "expected_document_ids": ["Beatmap"]}
```

The keyword evaluation is the notebook equivalent of the cluster `smoke` task.

In [ ]:
EVAL_DATASET = Path("eval/osu_seed.jsonl")
KEYWORD_EVAL_REPORT = ARTIFACT_DIR / "eval_seed_keyword_report.json"

run_osu_bot("eval", EVAL_DATASET, "--output", KEYWORD_EVAL_REPORT)
keyword_eval = show_json(KEYWORD_EVAL_REPORT, max_chars=3000)

In [ ]:
import pandas as pd

rows = keyword_eval["examples"]
df = pd.DataFrame(rows)
df[["category", "question", "matched", "top_document_ids"]].head(12)

## 10. Optional: Build a Dense Index

Dense indexing embeds chunks with `sentence-transformers/all-MiniLM-L6-v2` and writes vectors to local Qdrant storage.

Why start with a limit?

- Colab runtimes can disconnect.
- The first run has to download the embedding model.
- A small slice confirms the config works before you index everything.

Useful examples:

```python
run_osu_bot("index", "--limit", 500, "--batch-size", 32)
run_osu_bot("index", "--resume", "--limit", 2000, "--batch-size", 32)
```

If the small run works, increase `INDEX_LIMIT` or set it to `None` and remove the `--limit` argument in the code cell.

In [ ]:
RUN_DENSE_INDEXING = False
INDEX_LIMIT = 10000
INDEX_BATCH_SIZE = 64

if RUN_DENSE_INDEXING:
    start_osu_bot_job(
        "index",
        "index",
        "--resume",
        "--limit", INDEX_LIMIT,
        "--batch-size", INDEX_BATCH_SIZE,
    )
    print("Check progress with job_status('index') and show_job_log('index').")
else:
    print("Dense indexing skipped. Set RUN_DENSE_INDEXING = True to start it in the background.")

## 11. Optional: Dense Retrieval Evaluation

Run this only after building a dense index. Dense retrieval combines Qdrant results with the keyword/entity scoring pipeline.

This is the notebook equivalent of the cluster `eval_dense` task.

In [ ]:
RUN_DENSE_EVAL = False
DENSE_EVAL_REPORT = ARTIFACT_DIR / "eval_seed_dense_report.json"

if RUN_DENSE_EVAL:
    start_osu_bot_job("dense_eval", "eval", EVAL_DATASET, "--dense", "--output", DENSE_EVAL_REPORT)
    print("Check progress with job_status('dense_eval') and show_job_log('dense_eval').")
    print("After it finishes, run: dense_eval = show_json(DENSE_EVAL_REPORT, max_chars=3000)")
else:
    print("Dense evaluation skipped. Set RUN_DENSE_EVAL = True after indexing.")

## 12. Optional: GLiNER Entity Extraction

The deterministic `terms` command is the baseline. GLiNER is an experimental extra lane that proposes entity candidates from chunk text.

Start with a small limit. The default sampling mode is `balanced`, which scans across many documents instead of reading only the first chunks in artifact order.

Example command:

```bash
osu-bot --config config.colab.toml entities --backend gliner --label-profile main-page --limit 100 --sampling balanced --threshold 0.5
osu-bot --config config.colab.toml normalize-entities
```

The normalized review file is written to `artifacts/rag/entity_normalization_review.csv`.

In [ ]:
RUN_GLINER_ENTITIES = False
ENTITY_LIMIT = 100

if RUN_GLINER_ENTITIES:
    pre_command = f"{shlex.quote(sys.executable)} -m pip install -q -e '.[dev,entities]'"
    start_osu_bot_job(
        "entities",
        "entities",
        "--backend", "gliner",
        "--label-profile", "main-page",
        "--limit", ENTITY_LIMIT,
        "--sampling", "balanced",
        "--threshold", 0.5,
        pre_command=pre_command,
    )
    print("Check progress with job_status('entities') and show_job_log('entities').")
    print("After it finishes, run normalize-entities in the foreground or start another background job.")
else:
    print("GLiNER entity extraction skipped. Set RUN_GLINER_ENTITIES = True to start it in the background.")

## 13. About Answer Generation in Colab

The CLI command `query` expects a local Ollama server at `http://127.0.0.1:11434` and defaults to the `mistral` model.

That is convenient on a laptop, but less convenient in Colab because Colab sessions are temporary and do not ship with Ollama running.

For Colab, the most reliable workflow is:

1. Use `inspect` to verify retrieved sources.
2. Use `eval` to measure retrieval quality.
3. Download artifacts.
4. Run `query` locally later, or adapt the generation layer to call a hosted model API.

You can still run keyword-only inspection in Colab, which is the most important debugging step before answer generation.

In [ ]:
question = "What does osu!supporter unlock?"
run_osu_bot("inspect", question, "--keyword-only")

## 14. Save or Download Artifacts

Colab files disappear when the runtime is recycled. Download your artifacts before closing the session.

The zip file includes reports, JSONL artifacts, CSV review files, and the local Qdrant folder if you built a dense index.

In [ ]:
import zipfile

ZIP_PATH = Path("artifacts_rag_colab.zip")

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in ARTIFACT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.as_posix())

print(f"Wrote {ZIP_PATH.resolve()} ({ZIP_PATH.stat().st_size / 1_000_000:.2f} MB)")

if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))

## 15. Optional: Persist Artifacts to Google Drive

Downloading a zip is simple, but Drive is nicer for longer experiments. This cell mounts Drive and copies the artifact zip there.

You can also copy the whole `artifacts/rag` directory if you prefer to keep files uncompressed.

In [ ]:
COPY_TO_DRIVE = False

if COPY_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive mounting is only available in Colab.")
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/osu-chat-bot-artifacts")
    drive_dir.mkdir(parents=True, exist_ok=True)
    destination = drive_dir / ZIP_PATH.name
    shutil.copy2(ZIP_PATH, destination)
    print(f"Copied to {destination}")
else:
    print("Drive copy skipped. Set COPY_TO_DRIVE = True to enable it.")

## 16. Troubleshooting

### `Set REPO_URL...`

Colab needs to clone the project before it can install it. Set `REPO_URL` near the top of the notebook.

### `No chunks found. Run osu-bot ingest first.`

Run the baseline artifact section again. `inspect`, `query`, and `eval` need `chunks_hierarchical.jsonl`.

### `No terms found. Run osu-bot terms first.`

Run `terms`. Retrieval uses the term dictionary for entity-aware scoring.

### Dense retrieval fails

Make sure you ran `index` first. Keyword-only retrieval does not need Qdrant, but dense retrieval does.

### Hugging Face download is slow or throttled

Use a smaller `INDEX_LIMIT` while testing. For serious repeated runs, mount Drive and set `cache_folder` to a persistent Drive path.

### Colab disconnects

Use smaller indexing intervals, for example `--limit 500`, then resume with `--resume`. Download or copy artifacts to Drive between long stages.